<a href="https://colab.research.google.com/github/PTD504/flyrank-ai-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTD504/flyrank-ai-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane Choice:** Lane 1 — Freshness Decay / Decay Velocity Prediction.

**Why this lane?**

Organic search traffic naturally erodes over time as search intent evolves, competitors publish newer material, and content becomes outdated. The starter dataset provides rich historical trend metrics (impressions_last_30d vs impressions_prev_30d, trend_pct) alongside content staleness indicators (days_since_last_update, freshness_tier, content_age_days). Predicting decay velocity allows Flyrank and its clients to move from reactive content fixes to a proactive refresh strategy, intervening before rankings and traffic collapse completely.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**Research Question:**

*Which published pages have a high probability of severe organic traffic decay (trend_pct < -15%) in the upcoming period due to content staleness and declining engagement signals?*

- **Unit of Analysis:** A single piece of content / URL (content_id).
- **Model Output:** A Decay Risk Score ($0.0$ to $1.0$) indicating the probability of significant traffic loss, classified into risk tiers (High Decay Risk, Medium, Low).
- **Decision:** Deciding which specific URLs should be prioritized and queued for the upcoming Content Refresh Sprint, allocating limited editorial budget to pages where intervention prevents traffic loss.
- **Action Taken:** The SEO/Content team audits the flagged "High Decay Risk" URLs, updates outdated information, aligns content with current search intent, and re-optimizes internal linking.
- **Cost of a Wrong Call:**
    - **False Positive (Predicting decay when page is stable):** Wastes human editorial bandwidth and budget re-writing content that is performing well, with a secondary risk of accidentally disrupting currently stable rankings.
    - **False Negative (Missing a page undergoing severe decay):** Permanent loss of valuable organic search visibility and conversions to competitors, requiring significantly more resources to recover lost search rankings once a page drops out of the top results completely.
- **Why Data/ML helps:** A heuristic rule like "refresh all pages older than 1 year" fails because many old pages remain evergreen while others decay rapidly. ML captures complex non-linear relationships across content age, CTR changes, rank fluctuations (avg_position), and user engagement rates (engagement_rate, scroll_rate) to identify true decay signals early.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/PTD504/flyrank-ai-ml-internship.git

fatal: destination path 'flyrank-ai-ml-internship' already exists and is not an empty directory.


In [9]:
import pandas as pd

data_dir = "flyrank-ai-ml-internship/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_dir)

# 1. Metric 1: Share of content experiencing traffic decay ('down' trend)
down_count = (df['trend_direction'] == 'down').sum()
down_pct = (df['trend_direction'] == 'down').mean() * 100

print(f"[Metric 1] Declining Content Share ('down'): {down_pct:.2f}% ({down_count:,} out of {len(df):,} articles)")

# Metric 2: Mean Clicks & Active Click Rate to handle zero-heavy distribution
is_fresh = df['freshness_tier'] == '0-30'
is_stale = df['freshness_tier'] == '181+'

fresh_clicks_mean = df[is_fresh]['clicks_last_30d'].mean()
stale_clicks_mean = df[is_stale]['clicks_last_30d'].mean()

fresh_active_pct = (df[is_fresh]['clicks_last_30d'] > 0).mean() * 100
stale_active_pct = (df[is_stale]['clicks_last_30d'] > 0).mean() * 100

print(f"[Metric 2] Mean Clicks (Last 30d - Fresh vs Stale): {fresh_clicks_mean:.2f} vs {stale_clicks_mean:.2f}")
print(f"[Metric 2] Active Content Rate (Clicks > 0): {fresh_active_pct:.2f}% vs {stale_active_pct:.2f}%")

# 3. Metric 3: AI Traffic Penetration Rate
ai_traffic_articles_pct = (df['ai_sessions_90d'] > 0).mean() * 100
median_ai_share = df[df['ai_sessions_90d'] > 0]['ai_traffic_pct'].median()

print(f"[Metric 3] Content with AI Traffic: {ai_traffic_articles_pct:.2f}% of total articles (Median AI share: {median_ai_share:.2f}%)")

[Metric 1] Declining Content Share ('down'): 54.21% (16,262 out of 30,000 articles)
[Metric 2] Mean Clicks (Last 30d - Fresh vs Stale): 4.21 vs 0.57
[Metric 2] Active Content Rate (Clicks > 0): 34.68% vs 15.52%
[Metric 3] Content with AI Traffic: 6.43% of total articles (Median AI share: 2.86%)


## 4. Careful words: what I can and can't claim

### What this work CAN claim:
* **Observed Portfolio Trends:** Documented baseline metrics across 30,000 anonymized articles, revealing that 54.21% (16,262 articles) are in a declining traffic trend (`down`).
* **Directional Signal for Refresh Prioritization:** Established a clear performance divergence between freshness tiers (fresh `0-30` days articles show a 4.21 mean click rate and 34.68% active rate vs. 0.57 mean clicks and 15.52% active rate for stale `181+` days articles), providing an empirical baseline to prioritize refresh candidates.
* **AI Traffic Footprint:** Verified that AI-driven discovery is active within this dataset, with 6.43% of articles receiving non-zero AI sessions.

### What this work CANNOT claim:
* **Causal Proof:** Cannot prove that content age or lack of updates directly *causes* traffic decay. The observed drop is a correlation; external factors like seasonality, search intent evolution, or competitor actions are unobserved.
* **Google Algorithm Reverse-Engineering:** Cannot model, predict, or explain Google's underlying search ranking algorithm, as ranking mechanisms involve thousands of real-time signals beyond the 44 features captured in this snapshot.
* **Guaranteed Post-Refresh Performance:** Cannot guarantee that re-optimizing or refreshing a specific article will automatically recover lost traffic or increase rank position.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.